# Unit 1: What a conversation actually costs

The model remembers nothing. So the transcript is the memory. So the transcript goes
up the wire **on every single request**.

That one fact produces everything in this notebook.

Steps 2 and 5 run entirely offline and spend nothing, so they work even if your lane
is rate limited.

## Setup

In [1]:
import sys

from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / "src"))

In [2]:
from cse476.conversation import (
    count_tokens, transcript_tokens, tokenizer_is_exact,
    simulate_cost, sliding_window, sliding_window_with_pins,
    summarise_older, pin,
)
from cse476.lanes import get_client, MODEL, describe

print(describe())
print("tokenizer exact:", tokenizer_is_exact())

# WHY this check matters: tiktoken downloads its encoding file on first use.
# If that download failed, counts below are estimates, not measurements, and
# you should say so rather than quoting them as facts.
client = get_client()

Lane: Groq (groq, free)  |  Model: llama-3.3-70b-versatile


c:\Users\yashi\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


tokenizer exact: True


## 1. Tokens are not words

Try the three cases from the lecture. The third one is the interesting one.

In [3]:
samples = {
    "english prose": "Is there a room available at the Taj Palace on the fourteenth of August?",
    "json / schema": '{"type":"function","function":{"name":"get_room_availability"}}',
    "hindi":         "क्या चौदह अगस्त को ताज पैलेस में कमरा उपलब्ध है?",
}

for label, text in samples.items():
    print(f"{label:15} {len(text):>4} chars  {count_tokens(text):>4} tokens  "
          f"{count_tokens(text)/max(1,len(text.split())):.1f} tokens/word")

english prose     72 chars    16 tokens  1.1 tokens/word
json / schema     63 chars    14 tokens  14.0 tokens/word
hindi             48 chars    49 tokens  4.9 tokens/word


Look at the tokens per word column.

The same request costs meaningfully more in Hindi than in English, purely because of
how the tokenizer was built. That is a real fairness question about who pays more to
use the same product, and Unit 6 asks you to notice it rather than shrug.

## 2. The cost curve, offline

No API calls. Pure arithmetic. This is the shape of the problem.

In [4]:
rows = simulate_cost([50] * 40, system_tokens=100)

print(f"{'turn':>5} {'sent this request':>20} {'total so far':>15}")
for r in rows:
    if r.turn in (1, 5, 10, 20, 30, 40):
        print(f"{r.turn:>5} {r.sent:>20,} {r.cumulative:>15,}")

first, last = rows[0], rows[-1]
print()
print(f"40x the turns, but {last.cumulative/first.cumulative:.0f}x the total tokens.")

 turn    sent this request    total so far
    1                  150             150
    5                  350           1,250
   10                  600           3,750
   20                1,100          12,500
   30                1,600          26,250
   40                2,100          45,000

40x the turns, but 300x the total tokens.


Four times the turns costs roughly twelve times as much. Forty turns costs
several hundred times turn one.

Your conversation grows linearly. Your bill grows with the square.

## 3. Measure a real one

One line in your loop. The cheapest debugging tool in this course.

In [5]:
messages = [{"role": "system", "content": "You are a concise hotel booking assistant."}]

questions = [
    "My budget is Rs 7000 a night, remember that.",
    "What hotels do you know about?",
    "Which are under my budget?",
    "How far is the Radisson from campus?",
    "What is its guest rating?",
]

for n, q in enumerate(questions, 1):
    messages.append({"role": "user", "content": q})
    reply = client.chat.completions.create(model=MODEL, messages=messages)
    msg = reply.choices[0].message
    messages.append({"role": "assistant", "content": msg.content})
    print(f"turn {n}: sending {transcript_tokens(messages):>5} tokens")

turn 1: sending    56 tokens
turn 2: sending   196 tokens
turn 3: sending   369 tokens
turn 4: sending   443 tokens
turn 5: sending   542 tokens


The number never goes down. It cannot, because nothing removes anything.

## 4. Trim it

A sliding window keeps the system message and as many recent messages as fit.

In [6]:
before = transcript_tokens(messages)
trimmed = sliding_window(messages, max_tokens=220)
after = transcript_tokens(trimmed)

print(f"{before} tokens -> {after} tokens, {len(messages)} messages -> {len(trimmed)}")
print()
for m in trimmed:
    print(f"  {m['role']:10} {str(m.get('content'))[:65]}")

542 tokens -> 185 tokens, 11 messages -> 5

  system     You are a concise hotel booking assistant.
  user       How far is the Radisson from campus?
  assistant  I don't have specific information about a campus location. Could 
  user       What is its guest rating?
  assistant  The Radisson hotel typically has a guest rating of around 4-4.5 o


## 5. Now find what you broke

Ask it something only an early turn could answer.

**Do not read ahead. Run the cell and look at the answer.**

In [7]:
probe = trimmed + [{"role": "user", "content": "What was the budget I mentioned at the start?"}]
reply = client.chat.completions.create(model=MODEL, messages=probe)
print(reply.choices[0].message.content)

You didn't mention a budget at the start. I mistakenly introduced that detail later on. Your initial question was about the distance from the Radisson to a campus, and then you asked about the guest rating.


### Note how it failed

It probably did not say "I no longer have that information". It answered, fluently
and confidently, with something plausible.

**Trimming did not cause a crash. It caused a lie.** A crash appears in your logs. A
lie does not. That distinction is why Unit 5 spends a whole block on hallucination
detection.

## 6. Pin the things that must survive

One line of difference. Same budget, same data.

In [8]:
pinned = [
    {"role": "system", "content": "You are a concise hotel booking assistant."},
    pin({"role": "user", "content": "My budget is Rs 7000 a night, remember that."}),
]
for i in range(20):
    pinned.append({"role": "user", "content": f"filler question {i} about hotels " * 3})

naive = sliding_window(pinned, max_tokens=220)
safe  = sliding_window_with_pins(pinned, max_tokens=220)

def has_budget(ms):
    return any("budget" in (m.get("content") or "") for m in ms)

print(f"plain sliding window : budget survived = {has_budget(naive)}  "
      f"({transcript_tokens(naive)} tokens)")
print(f"with pinning         : budget survived = {has_budget(safe)}  "
      f"({transcript_tokens(safe)} tokens)")

plain sliding window : budget survived = False  (204 tokens)
with pinning         : budget survived = True  (197 tokens)


Same budget. Same conversation. One keeps the fact the whole task depends on and
one throws it away without telling you.

There is a test that proves exactly this in `tests/mock_run_l3.py`, scenarios 6 and 7,
and it runs offline.

## 7. Summarisation, the third option

Costs one extra model call. Keeps the gist, loses the exact wording.

In [9]:
def summarise(text: str) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarise this conversation in two sentences. Keep any numbers exactly."},
            {"role": "user", "content": text},
        ],
        max_tokens=120,
    )
    return r.choices[0].message.content or ""

compact = summarise_older(messages, keep_recent=4, summariser=summarise)
print(f"{transcript_tokens(messages)} -> {transcript_tokens(compact)} tokens")
print()
for m in compact:
    print(f"  {m['role']:10} {str(m.get('content'))[:70]}")

542 -> 312 tokens

  system     You are a concise hotel booking assistant.
  system     Summary of earlier conversation:
The user has a budget of Rs 7000 per 
  user       How far is the Radisson from campus?
  assistant  I don't have specific information about a campus location. Could you p
  user       What is its guest rating?
  assistant  The Radisson hotel typically has a guest rating of around 4-4.5 out of


## Your turn

**1. Add the measurement to your Practical 2 agent.** One print, every turn. Include
a screenshot of the growth in your submission.

**2. Choose a strategy and defend it.** Sliding window, summarisation, or pinning.
Write two sentences on why that one fits your agent, and one sentence on what it
loses.

**3. Pick your three pins.** For your agent, name three facts that must never be
dropped. The test is narrow: which facts, if silently lost, would make your agent
give a **confidently wrong** answer rather than an unhelpful one?

**4. Optional, and worth doing.** Set `max_tokens` low enough that your own agent
starts losing things, then find the smallest budget at which it still behaves
correctly. That number is a real engineering result about your own system.

In [10]:
# your work here
